# Experiment 3: representation, action channels, and whole-body support
Uses completed **Exp1** recordings for each checkpoint's **own-trajectory** representation quality, and **Exp2** separately for channel/support responses. Methods share experimental conditions, not realized transitions. No cross-policy replay, simulator, optimizer, training runner, or new collection. Representation cells do not require Exp2 recordings.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks/exp3_representation.py").exists())
sys.path.insert(0, str(ROOT / "notebooks"))
import exp3_representation as analysis

## Configuration
Completed Exp1 recordings under `recording_roots` are searched recursively for all available methods. Set `tasks` to a list of task IDs to select a subset, or set `methods` to an explicit mapping for checkpoint relocation/separate checkpoint cohorts (leave `tasks=None`). Exp2 and incomplete runs are excluded.
Use trusted checkpoints only. An omitted checkpoint uses the path recorded in metadata; an explicit path must have the same SHA256. Discovery never substitutes the latest trained model. Multiple checkpoints for one method require explicit selection/separate IDs. **All eligible native transitions are used by default**; budgets/quotas are ignored unless `balanced_resampling=True`. This optional mode is deterministic balanced **subsampling without replacement**, separately per checkpoint and loading range, not bootstrap resampling. Strict protocol checks may require narrower roots.

Fresh frozen inference is the default and must pass native parity. CPU/GPU precision may differ; do not increase tolerances just to suppress mismatches. `recorded_native_only=True` instead audits saved predictions, with no reinference parity claim. This choice is independent of full/subsampled selection. `inference_batch_size` bounds inference memory; trial sums/counts preserve exact aggregation across chunks.

In [ ]:
from b1z1_paper_names import DISPLAY_NAMES, LATEX_METHOD_NAMES
# Paper labels differ from implementation ablation numbers; task IDs stay unchanged.
CONFIG = {
    "recording_roots": [str(ROOT / "evaluation/exp1")],
    "tasks": None,  # All discovered tasks, or e.g. ["b1z1_pact", "b1z1_ppo_pos"].
    "methods": None,  # Optional explicit {ID: {"task": ..., "checkpoint": ...}}.
    "evaluation_seeds": [0, 1, 2],
    "conditions": ["nominal", "ee-disturbance", "combined-disturbance"],
    "balanced_resampling": False,  # False: ALL eligible native transitions; budgets/quotas ignored.
    "sampling_budget": None,  # Only when enabled: cap per checkpoint AND loading range.
    "samples_per_stratum": 32,  # Only when enabled and budget=None: condition/motion/phase/seed quotas.
    "inference_batch_size": 1024,  # Bounded inference batches across recording chunks.
    "dynamics_batch_size": 1024,  # Batch offline BARD; reduce if GPU memory is limited.
    "cpu_blas_threads": 1,  # Avoid thread-team overhead in small joint-space solves.
    "progress": True,  # Print each recording and throughput every 30 seconds.
    "cache_predictions": False,  # Optional minibatch caches, isolated by selection/prediction mode.
    "minimum_per_stratum": 8,  # Explicit diagnostics for under-covered observed strata.
    "exp2_roots": [str(ROOT / "evaluation/exp2")],
    "exp2_loading": ["loaded"],  # Add unloaded for separate matched-control outputs.
    "sampling_seed": 0,
    "device": "cuda:0",
    "matmul_precision": "high",  # Matches the inspected Isaac Lab recording; parity still required.
    "output_dir": str(ROOT / "analysis/exp3"),
    "dynamics": True,  # Direct BARD; requires CUDA, bard-pytorch-dynamics and Pinocchio.
    "urdf": None,  # Optional local relocation; must match recorded URDF hash.
    "recorded_native_only": False,
    "parity_atol": 2e-5, "parity_rtol": 2e-4,
    "latex_method_names": {},  # Optional overrides keyed by method ID; defaults resolved below.
}
if os.environ.get("EXP3_CONFIG"):
    CONFIG.update(json.loads(os.environ["EXP3_CONFIG"]))

CONFIG = analysis.resolve_methods(CONFIG)
display(pd.DataFrame.from_dict(CONFIG["methods"], orient="index"))
# Resolve paper labels/macros even when methods were supplied by JSON overrides.
for method in CONFIG["methods"].values():
    if method["task"] in DISPLAY_NAMES:
        method["display_name"] = DISPLAY_NAMES[method["task"]]
CONFIG["latex_method_names"] = {
    **{key: LATEX_METHOD_NAMES[method["task"]] for key, method in CONFIG["methods"].items()
       if method["task"] in LATEX_METHOD_NAMES},
    **CONFIG.get("latex_method_names", {}),
}


## Execute and audit
Each model consumes only its own checkpoint's recorded history, including native Pos torque-history and UniFP compensation commands. No compatibility grouping, invented FF channels, substituted actor actions, or cross-policy histories are used. Selection identities stream to JSONL; aggregate manifests record available/selected counts.

Current force heads are scored against the **previous applied interval load visible in history_t**. Newly imposed loads are labelled separately. GRFs use the recorded source torque-conditioning input and successor-yaw conversion, and are compared with raw interval-average world forces, not processed training labels.

In [ ]:
result = analysis.run_analysis(CONFIG)
for path in sorted(Path(result["output_dir"]).glob("native_*_table.csv")):
    display(pd.read_csv(path, header=[0, 1], index_col=0))
print("\n".join(result["diagnostics"]) or "No coverage diagnostics.")
display(pd.DataFrame(result["unavailable"]))

## Portable results
The page-wide tables contain EE force, torso force, torso moment, GRF, base-linear velocity, base-angular velocity, and joint-velocity component RMSE. Biases, individual axes/feet, arm/leg velocities, event phases and motion modes remain in CSV/JSON breakdowns. Table means first reduce within trials, then equally across evaluation seeds; SD uses ddof=1. Missing heads/dynamics are dashes, never zeros.

BARD uses q_t mechanics, actual interval-average executed torque, and frozen decoded loads for an analytic one-step velocity prediction. Measured-loading and constant-velocity references are separate variants. Native UniFP has no complete learned-loading dynamics result because it has no moment or GRF head. Physics losses being disabled in training does not disable these offline diagnostics.

In [ ]:
display(pd.DataFrame(result["per_seed"]).head(20))
display(pd.DataFrame(result["native_parity"].items(), columns=["Recording", "Native parity"]))
display(pd.DataFrame([r for r in result["table_summary"] if r["variant"] != 'decoded']))
display(pd.DataFrame(result["dynamics_approximations"]).head())
print("Manifest, native/reference diagnostics and native LaTeX tables:", result["output_dir"])
print("Selection:", result["selection_mode"], "eligible:", result["selection"]["available"], "selected:", result["selection"]["selected"])

## Action channels and whole-body support (Exp2)
Stream loaded pushing/pulling/carrying recordings for PILOT, UniFP, Pos, A1, A3, A6. Missing recordings/fields are reported, never collected. Dashed event lines use recorded timing. Loads/channels/support use interval timestamps; tracking uses successors. Bands show checkpoint SD when independently trained checkpoints exist, otherwise evaluation-seed SD; surviving trial counts are plotted.

Geometry is URDF FK of the **mean delayed position command** at measured base_t versus reference_t, an explicit approximation when substep targets vary. Torque RMS uses recorded substep squares; PD/FF are weighted motor-scaled requests, executed torques are clipped. Recorded CoM moments are averaged cross-products, not cross-products of means. Non-foot resultant contacts are included where recorded. Neither channel usage nor support moments uniquely identify compensation.

Offline dynamics include rotor armature and zero-target passive viscous damping. Unknown constraint-dependent friction torque and unrecorded stiffness-drive targets are omitted and labelled as approximations, not silently treated as measured zeros. Measured-loading versus decoded-loading errors help distinguish model mismatch from representation error. Support settling/overshoot are unavailable without a justified stationary reference.

In [ ]:
import exp3_support
support = exp3_support.analyze(CONFIG)
display(pd.DataFrame(support["summary"]))
display(pd.DataFrame(support["comparisons"]))
print("\n".join(support["diagnostics"]))
from IPython.display import Image
for path in sorted((Path(CONFIG["output_dir"]) / "action_support").glob("*.png")):
    display(Image(filename=str(path)))